# Financial Market Analytics & Decision Intelligence

This notebook is the analytical companion to the Flask dashboard. It focuses on transparent historical analysis, data quality, performance, volatility, drawdown, volume behaviour, and decision-oriented insights. Predictive modelling is deliberately secondary.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## 1. Load and validate the source data

The analysis uses the repository's historical OHLCV CSV. Dates are sorted before all time-series calculations.

In [ ]:
df = pd.read_csv('../data/stock_data.csv')
required = {'Date', 'Open', 'High', 'Low', 'Close', 'Volume'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=['Date', 'Open', 'High', 'Low', 'Close', 'Volume'])
df = df.drop_duplicates(subset='Date').sort_values('Date').reset_index(drop=True)

print(f'Observations: {len(df):,}')
print(f'Date range: {df.Date.min().date()} to {df.Date.max().date()}')
df.head()

In [ ]:
quality = pd.Series({
    'duplicate_dates': int(df['Date'].duplicated().sum()),
    'null_cells': int(df[['Date','Open','High','Low','Close','Volume']].isna().sum().sum()),
    'negative_values': int((df[['Open','High','Low','Close','Volume']] < 0).sum().sum()),
    'invalid_ohlc_rows': int(((df['High'] < df[['Open','Low','Close']].max(axis=1)) | (df['Low'] > df[['Open','High','Close']].min(axis=1))).sum())
})
quality.to_frame('count')

## 2. Analytical feature engineering

Returns are calculated only after chronological sorting. Rolling statistics use trailing observations, avoiding future information.

In [ ]:
df['Daily_Return'] = df['Close'].pct_change()
df['Cumulative_Return'] = df['Close'] / df['Close'].iloc[0] - 1
df['MA_20'] = df['Close'].rolling(20).mean()
df['MA_50'] = df['Close'].rolling(50).mean()
df['Rolling_Volatility_20D'] = df['Daily_Return'].rolling(20).std() * np.sqrt(252)
df['Rolling_Avg_Volume_20D'] = df['Volume'].rolling(20).mean()
df['Running_Peak'] = df['Close'].cummax()
df['Drawdown'] = df['Close'] / df['Running_Peak'] - 1
df['Volume_Change'] = df['Volume'].pct_change()
df['Range_Pct'] = (df['High'] - df['Low']) / df['Close']

df[['Date','Close','Daily_Return','MA_20','MA_50','Rolling_Volatility_20D','Drawdown']].tail()

## 3. Executive KPI summary

In [ ]:
kpis = pd.Series({
    'Latest close': df['Close'].iloc[-1],
    'Daily return': df['Daily_Return'].iloc[-1],
    'Period return': df['Cumulative_Return'].iloc[-1],
    'Annualized volatility': df['Daily_Return'].std() * np.sqrt(252),
    'Maximum drawdown': df['Drawdown'].min(),
    'Average volume': df['Volume'].mean(),
    'Up days': int((df['Daily_Return'] > 0).sum()),
    'Down days': int((df['Daily_Return'] < 0).sum())
})
kpis

## 4. Price trend and moving averages

In [ ]:
ax = df.plot(x='Date', y=['Close','MA_20','MA_50'], figsize=(12,5), title='Closing Price and Moving Averages')
ax.set_ylabel('Price')
plt.tight_layout()
plt.show()

## 5. Return and volatility behaviour

In [ ]:
ax = df.plot(x='Date', y='Rolling_Volatility_20D', figsize=(12,4), title='20-Day Rolling Annualized Volatility')
ax.set_ylabel('Volatility')
plt.tight_layout()
plt.show()

df['Daily_Return'].describe()

## 6. Drawdown and downside risk

In [ ]:
ax = df.plot(x='Date', y='Drawdown', figsize=(12,4), title='Historical Drawdown')
ax.axhline(0, linewidth=1)
ax.set_ylabel('Drawdown')
plt.tight_layout()
plt.show()

worst = df.nsmallest(10, 'Daily_Return')[['Date','Close','Daily_Return','Drawdown']]
worst

## 7. Volume behaviour and market activity

In [ ]:
df['Volume_Ratio_20D'] = df['Volume'] / df['Rolling_Avg_Volume_20D']
activity = df.nlargest(10, 'Volume_Ratio_20D')[['Date','Volume','Rolling_Avg_Volume_20D','Volume_Ratio_20D','Daily_Return']]
activity

## 8. Descriptive decision insights

These rules are descriptive signals, not investment recommendations. They help an analyst identify periods worth investigating.

In [ ]:
latest = df.iloc[-1]
signal = 'UP' if latest['Daily_Return'] > 0 else 'DOWN' if latest['Daily_Return'] < 0 else 'FLAT'
print(f"Current descriptive signal: {signal}")
print(f"Period return: {df['Cumulative_Return'].iloc[-1]:.2%}")
print(f"Maximum drawdown: {df['Drawdown'].min():.2%}")
print(f"Average daily return: {df['Daily_Return'].mean():.2%}")
print(f"Positive-day share: {(df['Daily_Return'] > 0).mean():.2%}")

## 9. Secondary predictive modelling check

The original notebook fitted the scaler before the chronological split, which leaks information from the test period. This version fixes that issue by splitting chronologically first and fitting the scaler only on training data. The model remains a small benchmark rather than the dashboard's primary analytical method.

In [ ]:
model_df = df[['Date','Open','High','Low','Close','Volume']].copy()
model_df['Target'] = model_df['Close'].shift(-1)
model_df = model_df.dropna().reset_index(drop=True)

feature_cols = ['Open','High','Low','Close','Volume']
split = int(len(model_df) * 0.8)
train = model_df.iloc[:split]
test = model_df.iloc[split:]

scaler = StandardScaler()
X_train = scaler.fit_transform(train[feature_cols])
X_test = scaler.transform(test[feature_cols])
y_train = train['Target']
y_test = test['Target']

model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f'Test MAE: {mae:.4f}')
print(f'Test RMSE: {rmse:.4f}')

## 10. Conclusions

- The dashboard should lead with historical performance, risk, volatility, volume, and drawdown analytics.
- Time-series transformations are chronological and use trailing windows.
- The predictive model is a benchmark only; it is not presented as a production forecasting engine.
- The analytical outputs support investigation and decision-making, not personalized financial advice.